Step 1: Load the 5-Day Splits and Establish the Seasonal Naïve Baseline
Run this cell to load your new partitions and evaluate the 5-day seasonal baseline:

In [1]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name in {"src", "notebooks"} else cwd
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load 5-day parquets
train = pd.read_parquet(DATA_DIR / "train_5d.parquet")
validation = pd.read_parquet(DATA_DIR / "validation_5d.parquet")
test = pd.read_parquet(DATA_DIR / "test_5d.parquet")

for frame in [train, validation, test]:
    frame["date"] = pd.to_datetime(frame["date"])

TARGET = "target_5d"
BASELINE_COL = "seasonal_naive_5d"
CATEGORICAL_FEATURES = ["item_id", "store_id"]
NUMERIC_FEATURES = [
    "day_of_week",
    "is_weekend",
    "month",
    "sell_price",
    "price_missing",
    "snap",
    "lag_0",
    "lag_7",
    "lag_14",
    "hist_sum_5d",
    "hist_mean_7d",
    "hist_mean_28d"
]
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

def calculate_metrics(name, y_true, y_pred, split_name="validation"):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    total_actual = np.sum(np.abs(y_true))
    wape = np.sum(np.abs(y_true - y_pred)) / total_actual if total_actual > 0 else np.nan
    return {
        "model": name,
        "split": split_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "WAPE": round(wape, 4)
    }

# Evaluate Seasonal Naive (prior week's 5-day sum)
val_naive = calculate_metrics("Seasonal Naive (5d Lag-7)", validation[TARGET], validation[BASELINE_COL], "validation")
test_naive = calculate_metrics("Seasonal Naive (5d Lag-7)", test[TARGET], test[BASELINE_COL], "test")

baseline_results_df = pd.DataFrame([val_naive, test_naive])
display(baseline_results_df)

,model,split,MAE,RMSE,WAPE
0,Seasonal Naive (5d Lag-7),validation,3.9855,8.2729,0.5823
1,Seasonal Naive (5d Lag-7),test,3.9414,7.9076,0.6110


Step 2: Train Candidate ML Pipelines & Compare on Validation
Train Linear Regression, Random Forest, and XGBoost (Tweedie) against the validation set:

In [2]:
# 1. Preprocessor builder
def make_pipeline(model, scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
        
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=numeric_steps), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
        ],
        remainder="drop"
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

# 2. Candidate models
candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_leaf=3, n_jobs=-1, random_state=42)
}

if XGBOOST_AVAILABLE:
    candidates["XGBoost (Tweedie)"] = XGBRegressor(
        objective="reg:tweedie",
        tweedie_variance_power=1.2,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )

X_train = train[FEATURE_COLUMNS]
y_train = train[TARGET]
X_val = validation[FEATURE_COLUMNS]
y_val = validation[TARGET]

trained_models = {}
val_results = [val_naive]

for name, estimator in candidates.items():
    print(f"Training {name} on 5-day cumulative target...")
    pipe = make_pipeline(estimator, scale_numeric=(name == "Linear Regression"))
    pipe.fit(X_train, y_train)
    
    val_preds = np.clip(pipe.predict(X_val), 0, None)
    trained_models[name] = pipe
    val_results.append(calculate_metrics(name, y_val, val_preds, "validation"))

val_comparison_df = pd.DataFrame(val_results).sort_values("WAPE").reset_index(drop=True)
display(val_comparison_df)

Training Linear Regression on 5-day cumulative target...
Training Random Forest on 5-day cumulative target...
Training XGBoost (Tweedie) on 5-day cumulative target...


,model,split,MAE,RMSE,WAPE
0,XGBoost (Tweedie),validation,3.0609,6.0979,0.4472
1,Random Forest,validation,3.1384,6.2113,0.4586
2,Linear Regression,validation,3.1394,6.2094,0.4587
3,Seasonal Naive (5d Lag-7),validation,3.9855,8.2729,0.5823


Step 3: Evaluate the Best Model on Held-Out Test Data
Confirm that the selected model beats the Seasonal Naïve baseline on unseen future test dates:

In [3]:
best_model_name = val_comparison_df.iloc[0]["model"]
best_pipeline = trained_models[best_model_name]

X_test = test[FEATURE_COLUMNS]
y_test = test[TARGET]

test_preds = np.clip(best_pipeline.predict(X_test), 0, None)
selected_test_result = calculate_metrics(best_model_name, y_test, test_preds, "test")

test_comparison_df = pd.DataFrame([selected_test_result, test_naive])
print(f"--- Held-out Test Set Performance (Selected: {best_model_name}) ---")
display(test_comparison_df)

--- Held-out Test Set Performance (Selected: XGBoost (Tweedie)) ---


,model,split,MAE,RMSE,WAPE
0,XGBoost (Tweedie),test,3.0803,6.0180,0.4775
1,Seasonal Naive (5d Lag-7),test,3.9414,7.9076,0.6110


Step 4: Train the P90 Safety Stock Model & Export ArtifactsNEXUS needs both the expected demand ($P_{50}$) for financial costing and an upper risk buffer ($P_{90}$) for stockout detection:

In [4]:
print("Training P90 Safety Stock Buffer Model...")
if XGBOOST_AVAILABLE:
    p90_estimator = XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=0.90,
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        tree_method="hist",
        random_state=42
    )
else:
    p90_estimator = RandomForestRegressor(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42)

p90_pipeline = make_pipeline(p90_estimator, scale_numeric=False)
p90_pipeline.fit(X_train, y_train)

# Save production artifacts for the FastAPI inference service
joblib.dump(best_pipeline, MODEL_DIR / "nexus_5d_mean_model.joblib")
joblib.dump(p90_pipeline, MODEL_DIR / "nexus_5d_p90_buffer.joblib")

print("Saved model artifacts:")
print(" -", MODEL_DIR / "nexus_5d_mean_model.joblib")
print(" -", MODEL_DIR / "nexus_5d_p90_buffer.joblib")

Training P90 Safety Stock Buffer Model...
Saved model artifacts:
 - c:\Akhil\AI_Engineer\2ndProject\nexus-forecasting\models\nexus_5d_mean_model.joblib
 - c:\Akhil\AI_Engineer\2ndProject\nexus-forecasting\models\nexus_5d_p90_buffer.joblib


Step 5: Generate the NEXUS Inventory Agent Shortage Feed
Test how the models feed directly into the downstream decision flow:

In [5]:
import uuid

# Sample current inventory balance per SKU-store (e.g. 5 units on hand)
unique_pairs = test[["item_id", "store_id"]].drop_duplicates().reset_index(drop=True)
np.random.seed(42)
unique_pairs["available_stock"] = np.random.randint(2, 20, size=len(unique_pairs))

# Simulate disruption detected at the latest date in the test split
latest_date = test["date"].max()
disruption_features = test[test["date"] == latest_date][FEATURE_COLUMNS].copy()

forecast_id = str(uuid.uuid4())
disruption_features["forecast_run_id"] = forecast_id
disruption_features["expected_5d_demand"] = np.round(best_pipeline.predict(disruption_features), 1)
disruption_features["p90_safety_demand"] = np.round(p90_pipeline.predict(disruption_features), 1)

# Merge with available inventory to identify actionable shortages
shortage_feed = disruption_features[
    ["forecast_run_id", "item_id", "store_id", "expected_5d_demand", "p90_safety_demand"]
].merge(unique_pairs, on=["item_id", "store_id"], how="inner")

shortage_feed["stockout_risk"] = shortage_feed["available_stock"] < shortage_feed["p90_safety_demand"]
shortage_feed["shortage_units"] = np.maximum(
    0, (shortage_feed["p90_safety_demand"] - shortage_feed["available_stock"]).round(0)
)

print(f"NEXUS Decision Feed generated for date {latest_date.date()} (Forecast Run ID: {forecast_id})")
display(shortage_feed[shortage_feed["stockout_risk"] == True].head(10))

NEXUS Decision Feed generated for date 2016-04-19 (Forecast Run ID: 13d82619-2b6b-4a5f-946f-565d0acdcb1d)


,forecast_run_id,item_id,store_id,expected_5d_demand,p90_safety_demand,available_stock,stockout_risk,shortage_units
10,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_002,CA_1,1.000000,3.300000,3,True,0.0
14,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_002,TX_1,0.400000,2.600000,2,True,1.0
30,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,CA_1,11.700000,16.600000,15,True,2.0
31,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,CA_2,16.799999,24.200001,19,True,5.0
32,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,CA_3,34.099998,37.500000,10,True,28.0
33,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,CA_4,4.700000,7.700000,3,True,5.0
35,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,TX_2,7.300000,11.200000,8,True,3.0
39,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_004,WI_3,4.200000,7.100000,4,True,3.0
42,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_005,CA_3,10.300000,16.700001,5,True,12.0
45,13d82619-2b6b-4a5f-946f-565d0acdcb1d,HOBBIES_1_005,TX_2,2.300000,6.100000,5,True,1.0
